# CellSight AI — External Validation (MetaboLights MTBLS8644)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix

FTP = "https://ftp.ebi.ac.uk/pub/databases/metabolights/studies/public/MTBLS8644/"

s = pd.read_csv(FTP + "s_MTBLS8644.txt", sep="\t", low_memory=False)
labels = s[["Sample Name", "Factor Value[Cohort]"]].dropna()
labels.columns = ["sample", "group"]
labels["group"] = labels["group"].astype(str).str.strip()
print("labels found:")
print(labels["group"].value_counts())

In [ ]:
m = pd.read_csv(FTP + "m_MTBLS8644_LC-MS_positive_reverse-phase_metabolite_profiling_v2_maf.tsv",
                sep="\t", low_memory=False)
meta_cols = ['database_identifier','chemical_formula','smiles','inchi','metabolite_identification',
             'mass_to_charge','retention_time','charge','fragmentation','modifications',
             'metabolite_assignment','taxid','species','database','database_version','reliability',
             'uri','search_engine','search_engine_score','smallmolecule_abundance_sub',
             'smallmolecule_abundance_stdev_sub','smallmolecule_abundance_std_error_sub']
sample_cols = [c for c in m.columns if c not in meta_cols]
data = m.set_index("metabolite_identification")[sample_cols].T
data.index.name = "sample"
data.columns = data.columns.astype(str)

d = data.merge(labels, left_index=True, right_on="sample").set_index("sample")
print("dataset:", d.shape[0], "samples x", d.shape[1]-1, "named metabolites")
print(d["group"].value_counts())
                                                
assert set(d["group"].unique()) == {"Healthy", "T2DM",
    "early stage (pre-diabetes)", "late stage (pre-diabetes)"}, \
    "labels did not load as expected — paste the value_counts above to your assistant"
print("self-check passed \u2705")

In [ ]:
def make_pipe():
    return Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=3000, class_weight="balanced", C=0.1))])

bin_df = d[d["group"].isin(["Healthy", "T2DM"])]
X1, y1 = bin_df.drop(columns="group"), (bin_df["group"] == "T2DM").astype(int)
assert y1.nunique() == 2, f"need 2 classes, got: {bin_df['group'].unique()}"

Xtr, Xte, ytr, yte = train_test_split(X1, y1, test_size=0.25, stratify=y1, random_state=42)
model_r = make_pipe().fit(Xtr, ytr)
print("TEST 1 — REPLICATION on independent cohort (different lab/machine/sample type)")
print("Test AUROC:", round(roc_auc_score(yte, model_r.predict_proba(Xte)[:, 1]), 3))
cv1 = cross_val_score(make_pipe(), X1, y1,
                      cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="roc_auc")
print("5-fold CV AUROC:", cv1.round(3), "| mean:", cv1.mean().round(3))

In [ ]:
d3 = d.copy()
d3["cat"] = d3["group"].map({"Healthy": "healthy",
                             "early stage (pre-diabetes)": "pre-diabetic",
                             "late stage (pre-diabetes)": "pre-diabetic",
                             "T2DM": "diabetic"})
X3, y3 = d3.drop(columns=["group", "cat"]), d3["cat"]

Xtr3, Xte3, ytr3, yte3 = train_test_split(X3, y3, test_size=0.25, stratify=y3, random_state=42)
model_3 = make_pipe().fit(Xtr3, ytr3)
auroc3 = roc_auc_score(yte3, model_3.predict_proba(Xte3), multi_class="ovr", average="macro")

print("TEST 2 — 3-class model")
print("Test macro-AUROC:", round(auroc3, 3))
cm = confusion_matrix(yte3, model_3.predict(Xte3), labels=["healthy", "pre-diabetic", "diabetic"])
print(pd.DataFrame(cm, index=["true healthy", "true pre-diabetic", "true diabetic"],
                      columns=["pred healthy", "pred pre-diabetic", "pred diabetic"]))

In [ ]:
for i, cls in enumerate(model_3.named_steps["clf"].classes_):
    coef = pd.Series(model_3.named_steps["clf"].coef_[i], index=X3.columns)
    print(f"\n== Strongest signals for '{cls}' ==")
    print(coef.sort_values(ascending=False).head(5).round(2).to_string())